## Project Objective

MarketSentinel is an **autonomous investment research agent** that monitors a
stock's price and volume, detects unusual movements, and investigates
*why* they happened. Every detected anomaly is classified as one of:

| Verdict | Meaning |
|---|---|
| `normal_market` | The move is explained by the overall market or sector (ordinary bull/bear behaviour). |
| `explained_by_news` | The move is stock-specific but has an identifiable catalyst (earnings, news). |
| `unexplained_suspicious` | A large stock-specific move, often with abnormal volume, and no identifiable explanation. |

> **Responsible-AI note:** the agent **cannot prove market manipulation**.
> Regulators need order-level trading data for that. The agent flags
> *unexplained* activity with a risk score and evidence, and uses cautious
> language throughout.


| | |
|---|---|
| **Team Leader** | Sanjay Kumar |
| **Team Members** | Sanjay Kumar, Meaha J |
| **GitHub Repository** | `https://github.com/meaha711/team2_usd`|
| **LLM** | Local model via Ollama or LM Studio (OpenAI-compatible API) |
| **Data** | Yahoo Finance (`yfinance`): prices, volume, benchmarks, earnings dates, news, insider filings |



## 1. Setup and Configuration

### Running the local model

**Option A: Ollama**
```bash
ollama pull llama3.1:8b      # or qwen2.5:7b, mistral, etc.
ollama serve                  # usually already running after install
```

**Option B: LM Studio.** Load any instruct model, open the **Developer**
tab and click **Start Server** (default `http://localhost:1234`). Then set
`LLM_PROVIDER = "lmstudio"` below.


In [22]:
import json
import os
import re
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path

In [23]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import Markdown, display

In [24]:
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

In [25]:
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama")

In [26]:
LLM_ENDPOINTS = {
    "ollama": "http://localhost:11434/v1",
    "lmstudio": "http://localhost:1234/v1",
}

In [27]:
LLM_BASE_URL = os.getenv("LLM_BASE_URL", LLM_ENDPOINTS[LLM_PROVIDER])

In [28]:
LLM_MODEL = os.getenv(
    "LLM_MODEL", "llama3.1:8b" if LLM_PROVIDER == "ollama" else ""
)

In [29]:
MEMORY_PATH = Path(os.getenv("AGENT_MEMORY_PATH", "agent_memory.json"))

In [30]:
RESET_MEMORY = True # True = clean demo; False = keep past learning

In [31]:
DEMO_MAX_EVENTS = 3  # anomalies investigated per run (LLM cost)

In [32]:
MARKET_ETF = "SPY"

In [33]:
WARMUP_DAYS = 150 # extra history for rolling baselines

In [34]:
VERDICTS = {
    "normal_market": "Normal market activity",
    "explained_by_news": "Explained by news / earnings",
    "unexplained_suspicious": "Unexplained / suspicious activity",
}

In [35]:
VERDICT_COLORS = {
    "normal_market": "#8c8c8c",
    "explained_by_news": "#2b6cb0",
    "unexplained_suspicious": "#c53030",
}

In [36]:
print("Configuration loaded. LLM provider:", LLM_PROVIDER, LLM_BASE_URL)

Configuration loaded. LLM provider: ollama http://localhost:11434/v1


## 2. Local LLM Client


In [37]:
THINK_TAG = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)

In [38]:
JSON_RULE = (
    "\n\nRespond with ONE valid JSON object only. No markdown, no code "
    "fences, no commentary before or after the JSON."
)

In [39]:
def strip_reasoning(text):
    """Remove <think> blocks emitted by reasoning models."""
    return THINK_TAG.sub("", text or "").strip()

In [40]:
def extract_json(text):
    """Return the first JSON object found in a model response."""
    text = re.sub(r"```(?:json)?", "", strip_reasoning(text))
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("no JSON object in model output")
    return json.loads(text[start:end + 1])

In [41]:
class LocalLLM:
    """Minimal client for an OpenAI-compatible local LLM server."""

    def __init__(self, base_url, model="", temperature=0.2, timeout=300):
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.temperature = temperature
        self.timeout = timeout
        self.stats = {"llm_calls": 0, "fallbacks": 0, "errors": 0}
        self.online = self._connect()

    def _connect(self):
        try:
            resp = requests.get(f"{self.base_url}/models", timeout=5)
            resp.raise_for_status()
            models = [m["id"] for m in resp.json().get("data", [])]
        except (requests.RequestException, ValueError, KeyError) as exc:
            print(f"[LLM] No local model reachable ({type(exc).__name__}). "
                  "Running in rule-based fallback mode.")
            return False
        if not self.model and models:
            self.model = models[0]
        if models and self.model not in models:
            print(f"[LLM] Warning: '{self.model}' not in {models}")
        print(f"[LLM] Connected to {self.base_url} | model = {self.model}")
        return bool(self.model)

    def chat(self, system, user, max_tokens=1500):
        """Send one chat request and return the raw text reply."""
        payload = {
            "model": self.model,
            "temperature": self.temperature,
            "max_tokens": max_tokens,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        }
        resp = requests.post(f"{self.base_url}/chat/completions",
                             json=payload, timeout=self.timeout)
        resp.raise_for_status()
        self.stats["llm_calls"] += 1
        return resp.json()["choices"][0]["message"]["content"] or ""

    def chat_json(self, system, user, fallback, validate=None,
                  max_tokens=1500):
        """Structured call: LLM JSON if valid, otherwise rule fallback."""
        if self.online:
            try:
                data = extract_json(
                    self.chat(system + JSON_RULE, user, max_tokens))
                if validate is None or validate(data):
                    data["_source"] = "llm"
                    return data
                self.stats["errors"] += 1
            except (requests.RequestException, ValueError, KeyError,
                    TypeError) as exc:
                self.stats["errors"] += 1
                print(f"[LLM] Invalid response, using rules: {exc}")
        self.stats["fallbacks"] += 1
        data = fallback()
        data["_source"] = "rules"
        return data

    def chat_text(self, system, user, fallback, max_tokens=2500):
        """Free-text call with a fallback. Returns (text, source)."""
        if self.online:
            try:
                text = strip_reasoning(self.chat(system, user, max_tokens))
                if len(text) > 40:
                    return text, "llm"
            except (requests.RequestException, KeyError) as exc:
                self.stats["errors"] += 1
                print(f"[LLM] Text call failed, using rules: {exc}")
        self.stats["fallbacks"] += 1
        return fallback(), "rules"

In [42]:
llm = LocalLLM(LLM_BASE_URL, LLM_MODEL)

[LLM] Connected to http://localhost:11434/v1 | model = llama3.1:8b
